# Generating Embeddings using Hugging Face (Free models)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU/GPU)

### What you will learn
1. What an embedding is (in simple words)
2. Installing free, open Hugging Face libraries
3. Generating embeddings with `sentence-transformers`
4. Comparing text similarity using cosine similarity
5. Visualizing embeddings in 2D
6. Building a mini Semantic Search engine (intermediate project)

> No API key needed. No paid model used. Everything here runs on the free Hugging Face open-source models.

## 1. What is an Embedding?

- An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text.
- Similar sentences → similar (close) vectors.
- Different sentences → far apart vectors.

Example:
- "I love dogs" and "I like puppies" → vectors will be **close**
- "I love dogs" and "The stock market crashed" → vectors will be **far apart**

We will use the `sentence-transformers` library, which uses free Hugging Face models.

### 2. Install Required Libraries
Run this once. `-q` keeps the output quiet (low token / clean output).

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib

### 3. Load a Free Hugging Face Model

We are using **`all-MiniLM-L6-v2`** — a small, fast, completely free open-source model from Hugging Face.
- Size: ~80MB
- Great for learning and even production use
- Output: 384-dimensional embedding vector per sentence

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the free pre-trained model from Hugging Face
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")

### 4. Generate Your First Embedding

In [ ]:
sentence = "I love learning about Artificial Intelligence."

embedding = model.encode(sentence)

print("Sentence:", sentence)
print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[:10])

**Discussion Point:** Notice the embedding has 384 numbers. These numbers don't mean anything individually — but together, they capture the *meaning* of the sentence.

### 5. Generate Embeddings for Multiple Sentences

In [ ]:
sentences = [
    "I love dogs",
    "I like puppies",
    "The stock market crashed today",
    "Cats are cute pets",
    "The economy is in recession"
]

embeddings = model.encode(sentences)

print("Number of sentences:", len(sentences))
print("Embedding matrix shape:", embeddings.shape)

### 6. Measure Similarity Between Sentences

We use **Cosine Similarity** to check how close two embeddings are.
- Score close to `1` → very similar meaning
- Score close to `0` → unrelated meaning
- Score close to `-1` → opposite meaning (rare with text)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

import pandas as pd
df = pd.DataFrame(similarity_matrix, index=sentences, columns=sentences)
df.round(2)

**Important Discussion Point:**
Look at the similarity score between *"I love dogs"* and *"I like puppies"* — it should be high.
Compare that to *"I love dogs"* vs *"The stock market crashed today"* — it should be low.

This is the **core idea behind semantic search, chatbots, and recommendation systems**.

### 7. Visualize Embeddings in 2D (using PCA)
Embeddings have 384 dimensions — we can't see that. So we reduce it to 2D just to *visualize* how sentences cluster.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(7, 5))
plt.scatter(reduced[:, 0], reduced[:, 1])

for i, sent in enumerate(sentences):
    plt.annotate(sent, (reduced[i, 0], reduced[i, 1]), fontsize=9)

plt.title("2D Visualization of Sentence Embeddings")
plt.xlabel("PCA Dimension 1")
plt.ylabel("PCA Dimension 2")
plt.grid(True)
plt.show()

**Discussion Point:** Sentences with similar meaning (dogs/puppies, stock/economy) should appear **closer together** in the plot, even though we never told the model to group them manually.

## 8. Intermediate Project: Mini Semantic Search Engine

**Goal:** Given a user query, find the most relevant sentence from a small database — using meaning, not just matching words.

In [ ]:
# Our small "document" database
documents = [
    "Python is a popular programming language for AI.",
    "The weather today is sunny and warm.",
    "Machine learning models learn patterns from data.",
    "I enjoy playing football on weekends.",
    "Deep learning uses neural networks with many layers.",
    "Paris is the capital city of France."
]

# Encode all documents once
doc_embeddings = model.encode(documents)

def semantic_search(query, top_k=2):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Rank documents by similarity score
    ranked_idx = scores.argsort()[::-1][:top_k]

    print(f"Query: {query}\n")
    for idx in ranked_idx:
        print(f"Score: {scores[idx]:.3f}  |  {documents[idx]}")

# Try it
semantic_search("Tell me about AI and neural networks")

In [ ]:
# Try another query
semantic_search("What is the capital of France?")

### 9. Key Takeaways (Recap for Students)

| Concept | Meaning |
|---|---|
| Embedding | Numeric vector representing meaning of text |
| Model used | `all-MiniLM-L6-v2` (free, open-source, Hugging Face) |
| Cosine Similarity | Measures how close two embeddings are |
| Semantic Search | Finding relevant text by meaning, not exact words |
| PCA | Used only to visualize high-dimension vectors in 2D |

### 10. Practice Exercises (for students)
1. Replace the `documents` list with your own 10 sentences and test `semantic_search`.
2. Try a different free model: `'all-mpnet-base-v2'` (more accurate, slightly slower) and compare results.
3. Build a simple FAQ bot: store 5 Q&A pairs, and retrieve the best matching answer for a new question.

## Practice Exercise Solutions

### Exercise 1: 10 custom documents

In [ ]:
documents = ["Photosynthesis converts sunlight into chemical energy.", "The cricket team chased down 300 runs.",
             "Credit card bills are due on the fifth.", "Quantum computers use qubits.",
             "The pasta was cooked al dente.", "A new metro line opens next year.",
             "Vaccines train the immune system.", "The guitarist played a long solo.",
             "Python decorators wrap functions.", "Monsoon rains arrive in June."]
doc_embeddings = model.encode(documents)
semantic_search("how do plants make food", top_k=2)
semantic_search("music performance", top_k=2)

### Exercise 2: MiniLM vs mpnet

In [ ]:
mpnet = SentenceTransformer("all-mpnet-base-v2")
mp_emb = mpnet.encode(documents)
for q in ["how do plants make food", "music performance"]:
    for name, m, emb in [("MiniLM", model, doc_embeddings), ("mpnet", mpnet, mp_emb)]:
        sc = cosine_similarity(m.encode([q]), emb)[0]
        top = sc.argsort()[::-1][:2]
        print(f"{q!r} [{name}]:", [(round(float(sc[i]), 3), documents[i][:30]) for i in top])

### Exercise 3: FAQ bot

In [ ]:
faqs = {
    "What is your return policy?": "Items can be returned within 15 days.",
    "How long does delivery take?": "Delivery takes 3-5 business days.",
    "Do you offer discounts for students?": "Yes, students get 10% off with a valid ID.",
    "How can I update my profile?": "Open Account > Profile > Edit.",
    "Is my data secure?": "We encrypt all data in transit and at rest.",
}
keys = list(faqs)
key_emb = model.encode(keys)

def faq_answer(user_q):
    sc = cosine_similarity(model.encode([user_q]), key_emb)[0]
    i = int(sc.argmax())
    return {"matched": keys[i], "score": round(float(sc[i]), 3), "answer": faqs[keys[i]]}

for q in ["how many days for shipping?", "any student offers?", "can I send things back?"]:
    print(faq_answer(q))